<h2>Description</h2>

L'objectif de ce code est de fusionner toutes les données, donc à savoir les données des champs Elysées avec les données externes. 
Nous voulons donc créer un dataframe qui contient :
<li>les informations météorologiques</li>
<li>les informations sur les vacances et jour fériés</li>
<li>les informations sur l'opération "Paris respire"</li>

In [85]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

doc = 'champs_elysees.csv'

print("Version installée de pandas : ")
print(pd.__version__)

Version installée de pandas : 
2.3.0


<h3>Analyse du dataset relatif à l'axe</h3>

In [86]:
df_axe = pd.read_csv("../datasets_axes_bruts/" + doc, sep=";")

# On convertit la date en format convenable 
df_axe['Date et heure de comptage'] = pd.to_datetime(df_axe['Date et heure de comptage'], 
                                      errors='coerce', utc=True).dt.tz_convert('Europe/Paris').dt.tz_localize(None)  

df_axe.head() 

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,Etat arc,Date debut dispo data,Date fin dispo data,geo_point_2d,geo_shape
0,4264,AV_Champs_Elysees,2024-12-09 05:00:00,199.0,2.20945,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Invalide,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
1,4264,AV_Champs_Elysees,2024-12-09 06:00:00,235.0,2.28778,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Invalide,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
2,4264,AV_Champs_Elysees,2024-12-09 09:00:00,1041.0,11.63222,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Invalide,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
3,4264,AV_Champs_Elysees,2025-09-02 09:00:00,1139.0,28.39222,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Ouvert,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
4,4264,AV_Champs_Elysees,2025-06-05 00:00:00,610.0,9.35833,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Ouvert,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."


In [87]:
df_axe.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval
count,8747.0,8747,8197.000000,8182.000000,8747.0,8747.0
mean,4264.0,2025-04-26 02:38:19.028238336,738.280224,15.296626,2294.0,2293.0
min,4264.0,2024-10-01 05:00:00,0.000000,0.000000,2294.0,2293.0
25%,4264.0,2025-01-22 16:30:00,532.000000,7.376945,2294.0,2293.0
50%,4264.0,2025-04-25 20:00:00,807.000000,15.152225,2294.0,2293.0
75%,4264.0,2025-08-06 21:30:00,944.000000,21.565000,2294.0,2293.0
max,4264.0,2025-11-07 00:00:00,2190.000000,77.545560,2294.0,2293.0
std,0.0,NaN,285.999446,9.220673,0.0,0.0


In [88]:
df_axe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8747 non-null   int64         
 1   Libelle                    8747 non-null   object        
 2   Date et heure de comptage  8747 non-null   datetime64[ns]
 3   Débit horaire              8197 non-null   float64       
 4   Taux d'occupation          8182 non-null   float64       
 5   Etat trafic                8747 non-null   object        
 6   Identifiant noeud amont    8747 non-null   int64         
 7   Libelle noeud amont        8747 non-null   object        
 8   Identifiant noeud aval     8747 non-null   int64         
 9   Libelle noeud aval         8747 non-null   object        
 10  Etat arc                   8747 non-null   object        
 11  Date debut dispo data      8747 non-null   object        
 12  Date f

In [89]:
df_axe['date'] = df_axe['Date et heure de comptage'].dt.date
df_axe['heure'] = df_axe['Date et heure de comptage'].dt.hour
df_axe['dow'] = df_axe['Date et heure de comptage'].dt.dayofweek
df_axe['mois'] = df_axe['Date et heure de comptage'].dt.month
df_axe['annee'] = df_axe['Date et heure de comptage'].dt.year
df_axe['jour_mois'] = df_axe['Date et heure de comptage'].dt.day
jour_map = {0:'Lun',1:'Mar',2:'Mer',3:'Jeu',4:'Ven',5:'Sam',6:'Dim'}
df_axe['jour_semaine'] = df_axe['dow'].map(jour_map)

In [90]:
fig = px.line(
    df_axe.sort_values('Date et heure de comptage'),
    x='Date et heure de comptage', y='Débit horaire',
    title=f"Débit horaire",
    labels={'Date et heure de comptage':'Date/Heure','Débit horaire':'Débit (véh/h)'}
)

fig.show()

In [91]:
profil = (df_axe
          .groupby(['jour_semaine','heure'], as_index=False)['Débit horaire']
          .mean())

fig = px.line(
    profil, x='heure', y='Débit horaire', color='jour_semaine',
    markers=True, title=f"Profil horaire moyen par jour",
    labels={'heure':'Heure','Débit horaire':'Débit moyen (véh/h)','jour_semaine':'Jour'}
)
fig.update_layout(template='simple_white')
fig.show()

In [92]:
heat = (df_axe
        .groupby(['jour_semaine','heure'], as_index=False)['Débit horaire']
        .mean())


heat['jour_semaine'] = pd.Categorical(heat['jour_semaine'],
                                      categories=['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'],
                                      ordered=True)

fig = px.imshow(
    heat.pivot(index='jour_semaine', columns='heure', values='Débit horaire').values,
    labels=dict(x="Heure", y="Jour", color="Débit moyen"),
    x=list(range(24)),
    y=['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'],
    title=f"Carte de chaleur"
)
fig.update_layout(template='simple_white')
fig.show()


In [93]:
df_scatter = df_axe.dropna(subset=['Débit horaire','Taux d\'occupation']).copy()
fig = px.scatter(
    df_scatter, x='Taux d\'occupation', y='Débit horaire',
    color='jour_semaine', opacity=0.7,
    title=f"Débit vs Taux d’occupation",
    labels={'Taux d\'occupation':'Taux d’occupation','Débit horaire':'Débit (véh/h)'}
)

fig.update_layout(template='simple_white')
fig.show()

In [94]:
box = df_axe.dropna(subset=['Débit horaire']).copy()
fig = px.box(
    box, x='jour_semaine', y='Débit horaire', points='all',
    title=f"Distribution du débit par jour",
    labels={'jour_semaine':'Jour','Débit horaire':'Débit (véh/h)'}
)
fig.update_layout(template='simple_white')
fig.show()


In [95]:
etat_jour = (df_axe
             .assign(jour=pd.to_datetime(df_axe['date']))
             .groupby(['jour','Etat trafic'], as_index=False)
             .size())

fig = px.area(
    etat_jour, x='jour', y='size', color='Etat trafic',
    title=f"État du trafic (comptes journaliers)",
    labels={'jour':'Date','size':'Occurrences'}
)
fig.update_layout(template='simple_white', hovermode='x unified')
fig.show()

In [96]:
mensuel = (df_axe
           .set_index('Date et heure de comptage')
           .resample('MS')['Débit horaire']
           .mean()
           .to_frame('debit_moyen')
           .reset_index())

mensuel['trend_3m'] = mensuel['debit_moyen'].rolling(3, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=mensuel['Date et heure de comptage'], y=mensuel['debit_moyen'],
    mode='lines+markers', name='Moyenne mensuelle'
))
fig.add_trace(go.Scatter(
    x=mensuel['Date et heure de comptage'], y=mensuel['trend_3m'],
    mode='lines', name='Tendance (MM-3)', line=dict(width=4)
))

fig.update_layout(
    title="Débit horaire – tendance mensuelle (moyenne + lissage 3 mois)",
    xaxis_title="Mois",
    yaxis_title="Débit moyen (véh/h)",
    template="simple_white",
    hovermode="x unified"
)
fig.show()


In [97]:
profil = (df_axe
          .groupby(['annee','mois'], as_index=False)['Débit horaire']
          .mean()
          .rename(columns={'Débit horaire':'debit_moyen'}))

fig = px.line(
    profil, x='mois', y='debit_moyen', color='annee',
    markers=True,
    title="Profil mensuel du débit par année",
    labels={'mois':'Mois', 'debit_moyen':'Débit moyen (véh/h)', 'annee':'Année'}
)
fig.update_layout(template='simple_white', xaxis=dict(dtick=1))
fig.show()


In [98]:
df_axe['heure_sin'] = np.sin(2 * np.pi * df_axe['heure'] / 24)
df_axe['heure_cos'] = np.cos(2 * np.pi * df_axe['heure'] / 24)

df_axe['jour_sin'] = np.sin(2 * np.pi * df_axe['dow'] / 7)
df_axe['jour_cos'] = np.cos(2 * np.pi * df_axe['dow'] / 7)

df_axe['mois_sin'] = np.sin(2 * np.pi * df_axe['mois'] / 12)
df_axe['mois_cos'] = np.cos(2 * np.pi * df_axe['mois'] / 12)

In [99]:
df_axe.head(10)

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,...,mois,annee,jour_mois,jour_semaine,heure_sin,heure_cos,jour_sin,jour_cos,mois_sin,mois_cos
0,4264,AV_Champs_Elysees,2024-12-09 05:00:00,199.0,2.20945,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,12,2024,9,Lun,0.965926,2.588190e-01,0.000000,1.000000,-2.449294e-16,1.000000e+00
1,4264,AV_Champs_Elysees,2024-12-09 06:00:00,235.0,2.28778,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,12,2024,9,Lun,1.000000,6.123234e-17,0.000000,1.000000,-2.449294e-16,1.000000e+00
2,4264,AV_Champs_Elysees,2024-12-09 09:00:00,1041.0,11.63222,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,12,2024,9,Lun,0.707107,-7.071068e-01,0.000000,1.000000,-2.449294e-16,1.000000e+00
3,4264,AV_Champs_Elysees,2025-09-02 09:00:00,1139.0,28.39222,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,9,2025,2,Mar,0.707107,-7.071068e-01,0.781831,0.623490,-1.000000e+00,-1.836970e-16
4,4264,AV_Champs_Elysees,2025-06-05 00:00:00,610.0,9.35833,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,6,2025,5,Jeu,0.000000,1.000000e+00,0.433884,-0.900969,1.224647e-16,-1.000000e+00
5,4264,AV_Champs_Elysees,2025-06-04 23:00:00,618.0,11.21278,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,6,2025,4,Mer,-0.258819,9.659258e-01,0.974928,-0.222521,1.224647e-16,-1.000000e+00
6,4264,AV_Champs_Elysees,2025-06-04 21:00:00,825.0,17.61167,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,6,2025,4,Mer,-0.707107,7.071068e-01,0.974928,-0.222521,1.224647e-16,-1.000000e+00
7,4264,AV_Champs_Elysees,2025-06-04 20:00:00,1028.0,23.76667,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,6,2025,4,Mer,-0.866025,5.000000e-01,0.974928,-0.222521,1.224647e-16,-1.000000e+00
8,4264,AV_Champs_Elysees,2025-06-04 17:00:00,937.0,21.89167,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,6,2025,4,Mer,-0.965926,-2.588190e-01,0.974928,-0.222521,1.224647e-16,-1.000000e+00
9,4264,AV_Champs_Elysees,2025-04-03 11:00:00,1138.0,16.94889,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,...,4,2025,3,Jeu,0.258819,-9.659258e-01,0.433884,-0.900969,8.660254e-01,-5.000000e-01


<h3>Nous insérons d'abord les données météorologiques</h3>

In [100]:
meteo = pd.read_csv("../datasets_externes_clean/meteo.csv", sep=";")
meteo['datetime'] = pd.to_datetime(meteo['datetime'])

In [101]:
df_merge = pd.merge(df_axe, meteo, left_on="Date et heure de comptage", right_on="datetime", how="left")

In [102]:
df_merge.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,jour_cos,mois_sin,mois_cos,Unnamed: 0,precipitations heure,duree prec (en min),force moyenne vent (m/s),Température,ensoleillement (en min),datetime
count,8747.0,8747,8197.000000,8182.000000,8747.0,8747.0,8747.000000,8747.000000,8747.000000,8747.000000,...,8747.000000,8.747000e+03,8.747000e+03,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747
mean,4264.0,2025-04-26 02:38:19.028238336,738.280224,15.296626,2294.0,2293.0,11.506574,2.994741,6.668000,2024.804047,...,-0.006239,-5.500528e-02,3.106604e-02,11546.638619,0.077867,3.980107,2.933686,13.218383,13.081685,2025-04-26 02:38:19.028238336
min,4264.0,2024-10-01 05:00:00,0.000000,0.000000,2294.0,2293.0,0.000000,0.000000,1.000000,2024.000000,...,-0.900969,-1.000000e+00,-1.000000e+00,6581.000000,0.000000,0.000000,0.000000,-3.600000,0.000000,2024-10-01 05:00:00
25%,4264.0,2025-01-22 16:30:00,532.000000,7.376945,2294.0,2293.0,6.000000,1.000000,4.000000,2025.000000,...,-0.900969,-8.660254e-01,-5.000000e-01,9304.500000,0.000000,0.000000,2.000000,8.700000,0.000000,2025-01-22 16:30:00
50%,4264.0,2025-04-25 20:00:00,807.000000,15.152225,2294.0,2293.0,12.000000,3.000000,7.000000,2025.000000,...,-0.222521,-2.449294e-16,6.123234e-17,11540.000000,0.000000,0.000000,2.800000,13.000000,0.000000,2025-04-25 20:00:00
75%,4264.0,2025-08-06 21:30:00,944.000000,21.565000,2294.0,2293.0,17.000000,5.000000,10.000000,2025.000000,...,0.623490,5.000000e-01,5.000000e-01,14013.500000,0.000000,0.000000,3.700000,17.700000,20.000000,2025-08-06 21:30:00
max,4264.0,2025-11-07 00:00:00,2190.000000,77.545560,2294.0,2293.0,23.000000,6.000000,12.000000,2025.000000,...,1.000000,1.000000e+00,1.000000e+00,16224.000000,14.300000,60.000000,9.300000,37.600000,60.000000,2025-11-07 00:00:00
std,0.0,NaN,285.999446,9.220673,0.0,0.0,6.919751,1.993350,3.434265,0.396955,...,0.707910,7.313334e-01,6.791720e-01,2769.396136,0.518787,12.900969,1.311747,6.678668,21.956183,NaN


In [103]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8747 non-null   int64         
 1   Libelle                    8747 non-null   object        
 2   Date et heure de comptage  8747 non-null   datetime64[ns]
 3   Débit horaire              8197 non-null   float64       
 4   Taux d'occupation          8182 non-null   float64       
 5   Etat trafic                8747 non-null   object        
 6   Identifiant noeud amont    8747 non-null   int64         
 7   Libelle noeud amont        8747 non-null   object        
 8   Identifiant noeud aval     8747 non-null   int64         
 9   Libelle noeud aval         8747 non-null   object        
 10  Etat arc                   8747 non-null   object        
 11  Date debut dispo data      8747 non-null   object        
 12  Date f

In [104]:
df_axe = df_merge

<h3>Nous allons maintenant ajouter les données relatives au jours piétonnisés</h3>

Nous allons donc ajouter une colonne de 1 ou 0 pour indiquer si oui ou non il s'agissait d'un jour piéton.

In [105]:
pieton = pd.read_csv("../datasets_externes_clean/pieton.csv", sep=";")
pieton.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  20 non-null     int64 
 1   date_debut  20 non-null     object
 2   date_fin    20 non-null     object
dtypes: int64(1), object(2)
memory usage: 608.0+ bytes


In [106]:
pieton['date_debut'] = pd.to_datetime(pieton['date_debut'])
pieton['date_fin'] = pd.to_datetime(pieton['date_fin'])

Nous écrivons ci-dessous une fonction pour déterminer si pour une date donnée, certains axes parisiens étaient piétonnisés.

In [107]:
def est_pietonnise(date):
    debut = pieton['date_debut']
    fin = pieton['date_fin']
    for i in range (len(debut)):
        if date >= debut[i] and date < fin[i]:
            return 1
    return 0

df_axe['est_pieton'] = df_axe['Date et heure de comptage'].apply(est_pietonnise)

In [108]:
df_axe[df_axe['est_pieton'] == 1][['Date et heure de comptage', 'est_pieton']]

,Date et heure de comptage,est_pieton
80,2024-11-03 16:00:00,1
81,2024-11-03 15:00:00,1
82,2024-11-03 12:00:00,1
83,2024-11-03 11:00:00,1
84,2024-11-03 10:00:00,1
...,...,...
6743,2025-09-21 16:00:00,1
6744,2025-09-21 14:00:00,1
6745,2025-09-21 10:00:00,1
6746,2025-09-21 08:00:00,1


In [109]:
df_axe = df_axe.sort_values(by="Date et heure de comptage", ascending=True)

<h3>On va maintenant ajouter des données sur les vacances, jours fériés...</h3>

In [110]:
vacances = pd.read_csv("../datasets_externes_clean/vacances.csv", sep=";")
vacances ['Date de début'] = pd.to_datetime(vacances['Date de début'])
vacances ['Date de fin'] = pd.to_datetime(vacances['Date de fin'])

In [111]:
def est_jour_vacances(dat):
    date_sans_heure = dat.date()
    debut = vacances['Date de début'].dt.date
    fin = vacances['Date de fin'].dt.date
    for i in range (len(debut)):
        if date_sans_heure >= debut[i] and date_sans_heure < fin[i]:
            return 1
    return 0

df_axe['est_vacances'] = df_axe['Date et heure de comptage'].apply(est_jour_vacances)

In [112]:
df = df_axe[df_axe['est_vacances'] == 1][['Date et heure de comptage', 'est_vacances']]

In [113]:
df.head(50)

,Date et heure de comptage,est_vacances
4796,2024-10-19 00:00:00,1
6434,2024-10-19 01:00:00,1
6433,2024-10-19 02:00:00,1
4961,2024-10-19 03:00:00,1
4960,2024-10-19 04:00:00,1
6432,2024-10-19 05:00:00,1
4959,2024-10-19 06:00:00,1
4958,2024-10-19 07:00:00,1
4957,2024-10-19 08:00:00,1
6431,2024-10-19 09:00:00,1


In [114]:
def est_avant_vacances(date):
    date_sans_heure = date.date()
    debut = vacances['Date de début'].dt.date
    for i in range (len(debut)):
        if date_sans_heure == debut[i] - pd.Timedelta(days=1):
            return 1
    return 0

df_axe['est_avant_vacances'] = df_axe['Date et heure de comptage'].apply(est_avant_vacances)

In [115]:
df_axe[df_axe['est_avant_vacances'] == 1][['est_avant_vacances', 'Date et heure de comptage']]

,est_avant_vacances,Date et heure de comptage
6161,1,2024-10-18 00:00:00
5963,1,2024-10-18 01:00:00
5962,1,2024-10-18 02:00:00
6138,1,2024-10-18 03:00:00
5961,1,2024-10-18 04:00:00
...,...,...
4297,1,2025-10-17 19:00:00
4298,1,2025-10-17 20:00:00
4017,1,2025-10-17 21:00:00
4018,1,2025-10-17 22:00:00


In [116]:
ferie = pd.read_csv("../datasets_externes_clean/ferie.csv", sep=";")

ferie['Date de début'] = pd.to_datetime(ferie['Date de début'], format='%d/%m/%Y')

In [117]:
def est_ferie(date):
    date = date.date()
    j_ferie = ferie['Date de début'].dt.date
    for el in j_ferie :
        if date == el:
            return 1
    return 0

df_axe['est_ferie'] = df_axe['Date et heure de comptage'].apply(est_ferie)

In [118]:
df_axe[df_axe['est_ferie']==1]['Date et heure de comptage']

8730   2024-11-01 00:00:00
7942   2024-11-01 01:00:00
8547   2024-11-01 02:00:00
8546   2024-11-01 03:00:00
2817   2024-11-01 04:00:00
               ...        
648    2025-11-01 19:00:00
605    2025-11-01 20:00:00
649    2025-11-01 21:00:00
606    2025-11-01 22:00:00
650    2025-11-01 23:00:00
Name: Date et heure de comptage, Length: 264, dtype: datetime64[ns]

In [119]:
def est_avant_ferie(date):
    date = date.date()
    j_ferie = ferie['Date de début'].dt.date
    for el in j_ferie :
        if date == el - pd.Timedelta(days=1):
            return 1
    return 0

df_axe['est_avant_ferie'] = df_axe['Date et heure de comptage'].apply(est_avant_ferie)

In [120]:
df_axe[df_axe['est_avant_ferie']==1]['Date et heure de comptage']

8621   2024-10-31 00:00:00
8746   2024-10-31 01:00:00
8745   2024-10-31 02:00:00
8723   2024-10-31 03:00:00
8722   2024-10-31 04:00:00
               ...        
8726   2025-10-31 19:00:00
8725   2025-10-31 20:00:00
8583   2025-10-31 21:00:00
8582   2025-10-31 22:00:00
8581   2025-10-31 23:00:00
Name: Date et heure de comptage, Length: 264, dtype: datetime64[ns]

In [121]:
df_axe['est_weekend'] = df_axe['dow'].isin([5, 6]).astype(int)

In [122]:
def est_rentree(date):
    date = date.date()  
    rentree = ['2025-08-28', '2025-08-29', '2025-08-30', '2025-08-31', '2025-09-01', '2025-09-02']
    rentree = [pd.to_datetime(d).date() for d in rentree]  
    
    return 1 if date in rentree else 0

df_axe['est_rentree'] = df_axe['Date et heure de comptage'].apply(est_rentree)

In [123]:
df_axe[df_axe['est_rentree']==1]['Date et heure de comptage']

7958   2025-08-28 00:00:00
3893   2025-08-28 01:00:00
3894   2025-08-28 02:00:00
8205   2025-08-28 03:00:00
8206   2025-08-28 04:00:00
               ...        
298    2025-09-02 19:00:00
385    2025-09-02 20:00:00
386    2025-09-02 21:00:00
387    2025-09-02 22:00:00
299    2025-09-02 23:00:00
Name: Date et heure de comptage, Length: 144, dtype: datetime64[ns]

Nous vérifions maintenant la composition du dataset

In [124]:
df_axe = df_axe.drop('Unnamed: 0', axis = 1)
df_axe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8747 entries, 1072 to 2034
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8747 non-null   int64         
 1   Libelle                    8747 non-null   object        
 2   Date et heure de comptage  8747 non-null   datetime64[ns]
 3   Débit horaire              8197 non-null   float64       
 4   Taux d'occupation          8182 non-null   float64       
 5   Etat trafic                8747 non-null   object        
 6   Identifiant noeud amont    8747 non-null   int64         
 7   Libelle noeud amont        8747 non-null   object        
 8   Identifiant noeud aval     8747 non-null   int64         
 9   Libelle noeud aval         8747 non-null   object        
 10  Etat arc                   8747 non-null   object        
 11  Date debut dispo data      8747 non-null   object        
 12  Date fin

In [125]:
df_axe.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,Température,ensoleillement (en min),datetime,est_pieton,est_vacances,est_avant_vacances,est_ferie,est_avant_ferie,est_weekend,est_rentree
count,8747.0,8747,8197.000000,8182.000000,8747.0,8747.0,8747.000000,8747.000000,8747.000000,8747.000000,...,8747.000000,8747.000000,8747,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000
mean,4264.0,2025-04-26 02:38:19.028238336,738.280224,15.296626,2294.0,2293.0,11.506574,2.994741,6.668000,2024.804047,...,13.218383,13.081685,2025-04-26 02:38:19.028238336,0.012804,0.356579,0.016577,0.030182,0.030182,0.282383,0.016463
min,4264.0,2024-10-01 05:00:00,0.000000,0.000000,2294.0,2293.0,0.000000,0.000000,1.000000,2024.000000,...,-3.600000,0.000000,2024-10-01 05:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4264.0,2025-01-22 16:30:00,532.000000,7.376945,2294.0,2293.0,6.000000,1.000000,4.000000,2025.000000,...,8.700000,0.000000,2025-01-22 16:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,4264.0,2025-04-25 20:00:00,807.000000,15.152225,2294.0,2293.0,12.000000,3.000000,7.000000,2025.000000,...,13.000000,0.000000,2025-04-25 20:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4264.0,2025-08-06 21:30:00,944.000000,21.565000,2294.0,2293.0,17.000000,5.000000,10.000000,2025.000000,...,17.700000,20.000000,2025-08-06 21:30:00,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,4264.0,2025-11-07 00:00:00,2190.000000,77.545560,2294.0,2293.0,23.000000,6.000000,12.000000,2025.000000,...,37.600000,60.000000,2025-11-07 00:00:00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,0.0,NaN,285.999446,9.220673,0.0,0.0,6.919751,1.993350,3.434265,0.396955,...,6.678668,21.956183,NaN,0.112436,0.479016,0.127688,0.171097,0.171097,0.450184,0.127254


<h2>Ajout Lag</h2>

In [126]:
def process_lags(df, N_day_lags, N_last_days_forbidden=0, sampling_rate=1):
    df["Date et heure de comptage"] = pd.to_datetime(
        df["Date et heure de comptage"], utc=True
    )
    df = df.sort_values("Date et heure de comptage").set_index(
        "Date et heure de comptage"
    )

    full_index = pd.date_range(df.index.min(), df.index.max(), freq="H")
    df = df.reindex(full_index)

    Lags = []
    for lag_day in range(N_last_days_forbidden + 1, N_day_lags + 1):
        for lag_hour in range(0, 24, sampling_rate):
            lag_name = f"lag_dh_{lag_day}_{lag_hour}"
            df[lag_name] = df["Débit horaire"].shift(lag_day * 24 + lag_hour)
            Lags.append(lag_name)
            lag_name = f"lag_or_{lag_day}_{lag_hour}"
            df[lag_name] = df["Taux d'occupation"].shift(lag_day * 24 + lag_hour)
            Lags.append(lag_name)

    df = df.reset_index().rename(columns={"index": "Date et heure de comptage"})
    return df, Lags

lags = process_lags(df_axe, 35, 3, 1)
print("=== Lags ===")
print(lags[1])
df_axe = lags[0]

df_axe.tail()

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5826/465476860.py:9: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5826/465476860.py:19: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5826/465476860.py:16: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5826/465476860.py:19: PerformanceWarning:

=== Lags ===
['lag_dh_4_0', 'lag_or_4_0', 'lag_dh_4_1', 'lag_or_4_1', 'lag_dh_4_2', 'lag_or_4_2', 'lag_dh_4_3', 'lag_or_4_3', 'lag_dh_4_4', 'lag_or_4_4', 'lag_dh_4_5', 'lag_or_4_5', 'lag_dh_4_6', 'lag_or_4_6', 'lag_dh_4_7', 'lag_or_4_7', 'lag_dh_4_8', 'lag_or_4_8', 'lag_dh_4_9', 'lag_or_4_9', 'lag_dh_4_10', 'lag_or_4_10', 'lag_dh_4_11', 'lag_or_4_11', 'lag_dh_4_12', 'lag_or_4_12', 'lag_dh_4_13', 'lag_or_4_13', 'lag_dh_4_14', 'lag_or_4_14', 'lag_dh_4_15', 'lag_or_4_15', 'lag_dh_4_16', 'lag_or_4_16', 'lag_dh_4_17', 'lag_or_4_17', 'lag_dh_4_18', 'lag_or_4_18', 'lag_dh_4_19', 'lag_or_4_19', 'lag_dh_4_20', 'lag_or_4_20', 'lag_dh_4_21', 'lag_or_4_21', 'lag_dh_4_22', 'lag_or_4_22', 'lag_dh_4_23', 'lag_or_4_23', 'lag_dh_5_0', 'lag_or_5_0', 'lag_dh_5_1', 'lag_or_5_1', 'lag_dh_5_2', 'lag_or_5_2', 'lag_dh_5_3', 'lag_or_5_3', 'lag_dh_5_4', 'lag_or_5_4', 'lag_dh_5_5', 'lag_or_5_5', 'lag_dh_5_6', 'lag_or_5_6', 'lag_dh_5_7', 'lag_or_5_7', 'lag_dh_5_8', 'lag_or_5_8', 'lag_dh_5_9', 'lag_or_5_9', 'lag_d

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5826/465476860.py:16: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5826/465476860.py:19: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/qc/5qn44xzs143g6s6sy04v51qm0000gn/T/ipykernel_5826/465476860.py:16: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.conca

,Date et heure de comptage,Identifiant arc,Libelle,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,...,lag_dh_35_19,lag_or_35_19,lag_dh_35_20,lag_or_35_20,lag_dh_35_21,lag_or_35_21,lag_dh_35_22,lag_or_35_22,lag_dh_35_23,lag_or_35_23
9639,2025-11-06 20:00:00+00:00,4264.0,AV_Champs_Elysees,876.0,19.25333,Pré-saturé,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,685.0,10.73833,637.0,9.47834,635.0,11.00222,767.0,17.51278,836.0,20.21000
9640,2025-11-06 21:00:00+00:00,4264.0,AV_Champs_Elysees,846.0,17.45389,Pré-saturé,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,495.0,7.18889,685.0,10.73833,637.0,9.47834,635.0,11.00222,767.0,17.51278
9641,2025-11-06 22:00:00+00:00,4264.0,AV_Champs_Elysees,791.0,13.97556,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,447.0,5.69945,495.0,7.18889,685.0,10.73833,637.0,9.47834,635.0,11.00222
9642,2025-11-06 23:00:00+00:00,4264.0,AV_Champs_Elysees,744.0,14.43445,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,299.0,3.56667,447.0,5.69945,495.0,7.18889,685.0,10.73833,637.0,9.47834
9643,2025-11-07 00:00:00+00:00,4264.0,AV_Champs_Elysees,685.0,10.88556,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,234.0,3.15278,299.0,3.56667,447.0,5.69945,495.0,7.18889,685.0,10.73833


In [127]:
df_axe.to_csv("../datasets_axes_with_all_features/" + doc , sep=";")